# Code Review Agent

A multi-agent code review system built with the OpenAI Agents SDK.

Three specialist agents (security, performance, style) review code in parallel,
then a manager agent synthesizes the final verdict.

I wanted this agent to help me further understand the course so i started with raw agents without struntured outputs/guardrails etc and gradually improved the flow 

In [ ]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace

load_dotenv(override=True)

In [ ]:
security_agent = Agent(
    name="Security Reviewer",
    instructions="You are a security focused code reviewer. "
    "Analyze the given code for security vulnerabilities such as "
    "SQL injection, hardcoded secrets, unsafe file operations, "
    "missing input validation, and insecure cryptography. "
    "Be specific about which lines are affected and suggest concrete fixes. "
    "If the code is secure, say so — don't invent problems.",
    model="gpt-4o-mini",
)

In [ ]:
# Some deliberately flawed code to review

sample_code = '''
import sqlite3
import hashlib

DB_PASSWORD = "admin123"

def get_user(username):
    conn = sqlite3.connect("users.db")
    query = f"SELECT * FROM users WHERE username = '{username}'"
    result = conn.execute(query).fetchall()
    conn.close()
    return result

def hash_password(password):
    return hashlib.md5(password.encode()).hexdigest()

def read_file(filename):
    path = "/data/" + filename
    f = open(path, "r")
    data = f.read()
    return data
'''

In [ ]:
with trace("Security Review"):
    result = await Runner.run(security_agent, sample_code)
    print(result.final_output)

In [ ]:
from pydantic import BaseModel, Field


class ReviewFinding(BaseModel):
    line_reference: str = Field(description="The line or section of code this finding refers to")
    issue: str = Field(description="A clear description of the issue found")
    suggestion: str = Field(description="A concrete suggestion for how to fix it")
    severity: str = Field(description="One of: critical, warning, info")


class SpecialistReview(BaseModel):
    domain: str = Field(description="The review domain, e.g. security")
    findings: list[ReviewFinding] = Field(description="List of issues found in the code")
    score: int = Field(description="Overall score from 1 (poor) to 10 (excellent) for this domain")
    summary: str = Field(description="A 1-2 sentence summary of the review")

In [ ]:
security_agent = Agent(
    name="Security Reviewer",
    instructions="You are a security focused code reviewer. "
    "Analyze the given code for security vulnerabilities such as "
    "SQL injection, hardcoded secrets, unsafe file operations, "
    "missing input validation, and insecure cryptography. "
    "Be specific about which lines are affected and suggest concrete fixes. "
    "If the code is secure, say so — don't invent problems.",
    model="gpt-4o-mini",
    output_type=SpecialistReview,
)

In [ ]:
with trace("Structured Security Review"):
    result = await Runner.run(security_agent, sample_code)

review = result.final_output
print(f"Domain: {review.domain}")
print(f"Score: {review.score}/10")
print(f"Summary: {review.summary}\n")

for i, finding in enumerate(review.findings, 1):
    print(f"--- Finding {i} [{finding.severity}] ---")
    print(f"  Line: {finding.line_reference}")
    print(f"  Issue: {finding.issue}")
    print(f"  Fix: {finding.suggestion}\n")

In [ ]:
performance_agent = Agent(
    name="Performance Reviewer",
    instructions="You are a performance focused code reviewer. "
    "Analyze the given code for performance issues such as "
    "unnecessary loops, O(n^2)+ complexity, redundant computations, "
    "memory leaks, N+1 query patterns, blocking I/O in async contexts, "
    "and inefficient string concatenation. "
    "Be specific about which lines are affected and suggest concrete fixes. "
    "If the code is performant, say so  don't invent problems.",
    model="gpt-4o-mini",
    output_type=SpecialistReview,
)

style_agent = Agent(
    name="Style Reviewer",
    instructions="You are a code style and readability reviewer. "
    "Analyze the given code for style issues such as "
    "unclear variable names, functions doing too many things, "
    "missing comments on complex logic, inconsistent naming, "
    "dead code, magic numbers, and poor error handling. "
    "Be specific about which lines are affected and suggest concrete fixes. "
    "If the code is clean, say so  don't invent problems.",
    model="gpt-4o-mini",
    output_type=SpecialistReview,
)

In [ ]:
import asyncio

with trace("Parallel Reviews"):
    results = await asyncio.gather(
        Runner.run(security_agent, sample_code),
        Runner.run(performance_agent, sample_code),
        Runner.run(style_agent, sample_code),
    )

for r in results:
    review = r.final_output
    print(f"=== {review.domain.upper()} (Score: {review.score}/10) ===")
    print(f"{review.summary}\n")
    for i, finding in enumerate(review.findings, 1):
        print(f"  [{finding.severity}] {finding.issue}")
        print(f"    Fix: {finding.suggestion}\n")

In [ ]:
security_tool = security_agent.as_tool(
    tool_name="security_reviewer",
    tool_description="Review code for security vulnerabilities"
)
performance_tool = performance_agent.as_tool(
    tool_name="performance_reviewer",
    tool_description="Review code for performance issues"
)
style_tool = style_agent.as_tool(
    tool_name="style_reviewer",
    tool_description="Review code for style and readability issues"
)

In [ ]:
review_manager = Agent(
    name="Code Review Manager",
    instructions="""You are a senior engineering lead conducting a code review.

Follow these steps:
1. Send the code to ALL THREE reviewer tools (security_reviewer, performance_reviewer, style_reviewer).
   Do not skip any reviewer.

2. Once all three reviews are back, synthesize a final review report in this format:

   ## Code Review Summary
   (2-3 sentence overall assessment)

   ## Security (Score: X/10)
   (Key findings)

   ## Performance (Score: X/10)
   (Key findings)

   ## Style (Score: X/10)
   (Key findings)

   ## Critical Issues
   (List any critical-severity items — if none, say "None found")

   ## Recommended Changes
   (Prioritized list of suggested improvements)

   ## Overall Score: X/10
   (Weighted average — security counts double)

Rules:
- You MUST use all three reviewer tools  do not write the review yourself.
- Be honest: if the code is good, say so.
- Prioritize critical issues first in recommendations.""",
    tools=[security_tool, performance_tool, style_tool],
    model="gpt-4o-mini",
)

In [ ]:
from IPython.display import display, Markdown

with trace("Managed Code Review"):
    result = await Runner.run(review_manager, sample_code)

display(Markdown(result.final_output))

In [ ]:
from agents import input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered


class CodeCheckOutput(BaseModel):
    is_code: bool = Field(description="Whether the input contains source code")
    reason: str = Field(description="Brief explanation")


code_check_agent = Agent(
    name="Code Checker",
    instructions="Determine if the user's input contains source code in any programming language. "
    "Look for function definitions, variable assignments, imports, loops, conditionals, "
    "class definitions, or other programming constructs. "
    "A code snippet with minor surrounding explanation still counts as code.",
    output_type=CodeCheckOutput,
    model="gpt-4o-mini",
)


@input_guardrail
async def code_input_guardrail(ctx, agent, message):
    result = await Runner.run(code_check_agent, message, context=ctx.context)
    return GuardrailFunctionOutput(
        output_info={"is_code": result.final_output.is_code, "reason": result.final_output.reason},
        tripwire_triggered=not result.final_output.is_code,
    )

In [ ]:
# Recreate the manager with the guardrail attached

review_manager = Agent(
    name="Code Review Manager",
    instructions="""You are a senior engineering lead conducting a code review.

Follow these steps:
1. Send the code to ALL THREE reviewer tools (security_reviewer, performance_reviewer, style_reviewer).
   Do not skip any reviewer.

2. Once all three reviews are back, synthesize a final review report in this format:

   ## Code Review Summary
   (2-3 sentence overall assessment)

   ## Security (Score: X/10)
   (Key findings)

   ## Performance (Score: X/10)
   (Key findings)

   ## Style (Score: X/10)
   (Key findings)

   ## Critical Issues
   (List any critical-severity items — if none, say "None found")

   ## Recommended Changes
   (Prioritized list of suggested improvements)

   ## Overall Score: X/10
   (Weighted average — security counts double)

Rules:
- You MUST use all three reviewer tools — do not write the review yourself.
- Be honest: if the code is good, say so.
- Prioritize critical issues first in recommendations.""",
    tools=[security_tool, performance_tool, style_tool],
    model="gpt-4o-mini",
    input_guardrails=[code_input_guardrail],
)

In [ ]:


try:
    with trace("Guardrail Test"):
        result = await Runner.run(review_manager, "What is the meaning of life?")
        print(result.final_output)
except InputGuardrailTripwireTriggered:
    print("Guardrail triggered: input rejected — please provide source code to review.")

In [ ]:


with trace("Guarded Code Review"):
    result = await Runner.run(review_manager, sample_code)

display(Markdown(result.final_output))

In [ ]:
# Paste any code snippet here to get a full review

your_code = '''
# Replace this with your own code
'''

with trace("Custom Code Review"):
    result = await Runner.run(review_manager, your_code)

display(Markdown(result.final_output))